# P69 — Un modelo de razonamiento inexacto en medicina

## 1. Título y paper

**Paper:** *A Model of Inexact Reasoning in Medicine*  
**Autoría:** Edward H. Shortliffe, Bruce G. Buchanan  
**Año y venue:** 1975 · Mathematical Biosciences, 23(3–4), 351–379  
**Nivel:** L2 · **Motor:** `mycin`  
**Ficha completa:** [`P69_mycin`](../../papers/foundational/P69_mycin/README.md)

**Hito:** El motor de MYCIN: razonar con grados de creencia y explicar cada conclusión por las reglas que la sostienen.

- [doi:10.1016/0025-5564(75)90047-4](https://doi.org/10.1016/0025-5564(75)90047-4)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El conocimiento médico está lleno de indicios que no son ni ciertos ni falsos. Aplicar probabilidad bayesiana exigía distribuciones conjuntas que nadie podía estimar ni declarar.
2. Ejecutar una implementación mínima de la propuesta: Los factores de certeza: un número en [−1, 1] por regla, con un álgebra de combinación que satura y admite evidencia en contra, más una traza que hace explicable cada conclusión.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Feigenbaum et al. (1971), DENDRAL
- P66


## 4. Intuición

Un médico no dice «es una infección por E. coli» ni «no lo es»: dice «bastante probable, por estos tres indicios». MYCIN reproduce eso con un número por regla y un álgebra para combinarlos, y con algo que resultó ser tan importante como el número: la lista de reglas que sostienen cada conclusión.


## 5. Concepto mínimo

```text
Regla:  SI premisas ENTONCES conclusión, con CF ∈ [−1, 1]

Disparo:      CF(conclusión) = min(CF de las premisas) × CF(regla)
Dos a favor:  CF = a + b·(1 − a)          ← satura, nunca llega a 1
Dos en contra:CF = a + b·(1 + a)
Mezcladas:    CF = (a + b) / (1 − min(|a|, |b|))
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('mycin', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Con qué grado sale la conclusión `enterobacteria`?
2. ¿Se llega a la certeza acumulando indicios a favor?
3. ¿Qué aporta la regla con factor negativo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('mycin', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('mycin', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

`enterobacteria` sale con **0,933** y `e_coli` con **0,701**: el sistema no responde sí o no. Y no se llega a la certeza: `a + b(1−a)` satura por debajo de 1 por muchos indicios que se acumulen. La regla negativa resta en el mismo eje, que es una decisión de diseño sin equivalente directo en probabilidad.


## 10. Comentario pedagógico

Lo que hizo aceptable a MYCIN entre médicos no fue su exactitud —que era comparable a la de los especialistas— sino que **podía explicar cada conclusión**. Cincuenta años después esa sigue siendo la ventaja estructural del razonamiento simbólico frente a un modelo denso, y la razón de que [P72](../../papers/foundational/P72_neurosimbolico/README.md) proponga combinarlos.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer los factores de certeza como probabilidades.


In [ ]:
print('Un CF de 0,7 no es una probabilidad de 0,7.')
print('Su algebra no se deriva de los axiomas de Kolmogorov ni respeta la regla de Bayes.')
print('Es un formalismo pragmatico de 1975, y sus propios autores lo revisaron despues.')

## 12. Corrección

La comparación honesta con la alternativa probabilística:


In [ ]:
r = run_paper_lab('mycin', seed=7)['result']
print('combinacion MYCIN de 0,7 y 0,5 :', r['combinacion_dos_evidencias_a_favor']['resultado'])
print('misma cuenta con independencia :', r['misma_cuenta_con_probabilidades_independientes'])
print('Coinciden en ESTE caso. No coinciden en general: no es una equivalencia.')

## 13. Desafío guiado

Sigue la traza de inferencia e identifica qué regla aporta cada incremento al factor de `enterobacteria`.


In [ ]:
r = run_paper_lab('mycin', seed=3)['result']
show(r)

## 14. Desafío autónomo

Escribe una base de diez reglas para un dominio que conozcas, con sus factores de certeza, y prueba a cambiar el orden de disparo. Después documenta qué conclusiones cambian y por qué no deberían.


## 15. Evidencia de aprendizaje

Guarda la traza con el aporte de cada regla y tu comparación entre la combinación de MYCIN y la probabilística.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P69_mycin/README.md) · evaluación formal: [`assessments/papers/P69_mycin.md`](../../assessments/papers/P69_mycin.md)


## 16. Cierre

Las reglas ya manejan incertidumbre. Queda el problema inverso: cuando las restricciones son duras y muchas, conviene podar antes de empezar a buscar.


## 17. Conexión con el siguiente hito

- P52
- P72

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
